In [1]:
import pandas as pd

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

In [2]:
train_df.head()

,id,path,question,a,b,c,d,answer
0,train_0001,train/train_0001.jpg,이 사진 속 운동기구가 설치된 장소는 어디일까요?,학교 운동장,공원,헬스장 내부,쇼핑몰 내부,b
1,train_0002,train/train_0002.jpg,이 사진에 보이는 전통 한국 건축물은 무엇인가요?,궁궐,성,사찰,한옥,d
2,train_0003,train/train_0003.jpg,이 사진에서 보이는 탈것은 무엇인가요?,세발 오토바이,오토바이,자동차,자전거,a
3,train_0004,train/train_0004.jpg,이 사진에서 보이는 새는 무엇인가요?,참새,갈매기,백로,오리,c
4,train_0005,train/train_0005.jpg,이 사진에서 사람들이 모여서 보고 있는 것은 무엇인가요?,해수욕,바다 축제,불꽃놀이,해돋이,d


In [3]:
test_df.head()

,id,path,question,a,b,c,d
0,test_0001,test/test_0001.jpg,이 사진에 보이는 탑은 무엇인가요?,63빌딩,남산타워,롯데타워,한강대교
1,test_0002,test/test_0002.jpg,이 사진에서 볼 수 있는 나무의 잎 색깔은 무엇인가요?,초록색,노란색,파란색,빨간색
2,test_0003,test/test_0003.jpg,이 사진에 보이는 건물은 무엇인가요?,창덕궁 인정전,경복궁 광화문,덕수궁 석조전,종묘 정전
3,test_0004,test/test_0004.jpg,이 사진에서 보이는 신호등의 상태는 무엇인가요?,파란불,깜빡이는 불,빨간불,노란불
4,test_0005,test/test_0005.jpg,사진 속 의자 위에 놓여 있지 않은 것은 무엇인가요?,담요,잡지,노트북,뜨개 인형


---

In [4]:
from transformers import pipeline  # Hugging Face 파이프라인

!pip -qq install googletrans==4.0.0-rc1
from googletrans import Translator  # googletrans 라이브러리

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langsmith 0.4.37 requires httpx<1,>=0.23.0, but you have httpx 0.13.3 which is incompatible.
firebase-admin 6.9.0 requires httpx[http2]==0.28.1, but you have httpx 0.13.3 which is incompatible.
openai 1.109.1 requires httpx<1,>=0.23.0, 

In [5]:
import pandas as pd
import re

In [6]:
def naive_detect_lang(text):
    # 한글이 있으면 한국어
    if re.search(r'[ㄱ-ㅎㅏ-ㅣ가-힣]', text):
        return 'ko'

    # 한글이 없으면 몰?루
    else:
        return 'other'

In [7]:
translator_hf = pipeline("translation_ko_to_en", model="Helsinki-NLP/opus-mt-ko-en")
translator_gg = Translator()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use cuda:0


In [8]:
# 제미나이가 짜준 GPU 배치 활용 코드

def batch_translate_dataframe(df, hf_translator, gg_translator):
    """
    DataFrame을 입력받아 'ko'는 HF 배치로, 'other'는 GG로 번역 후
    순서가 보장된 원본 형태의 DataFrame으로 반환합니다.
    """
    cols_to_translate = ['question', 'a', 'b', 'c', 'd']

    # 1. 번역할 텍스트만 stack()으로 쌓아 하나의 Series로 만듭니다.
    #    인덱스에 (원본 행 인덱스, 컬럼명)이 저장되어 위치를 기억합니다.
    texts_stacked = df[cols_to_translate].stack()

    # 2. 각 텍스트에 대해 언어 탐지
    lang_series = texts_stacked.apply(naive_detect_lang)

    # 3. 한국어(ko)와 그 외(other) 텍스트로 분리
    ko_texts_series = texts_stacked[lang_series == 'ko']
    other_texts_series = texts_stacked[lang_series == 'other']

    print(f"총 {len(texts_stacked)}개 텍스트 중")
    print(f"  > 한국어 (HF 번역): {len(ko_texts_series)}개")
    print(f"  > 그 외 (GG 번역): {len(other_texts_series)}개")

    # 4. (분리 1) 한국어 텍스트: 리스트로 만들어 HF 모델로 *배치 번역*
    translated_ko_list = []
    if not ko_texts_series.empty:
        ko_texts_list = ko_texts_series.tolist()

        # Hugging Face 파이프라인은 리스트를 받아 배치 처리합니다.
        hf_results = hf_translator(ko_texts_list, batch_size=64, truncation=True)
        translated_ko_list = [result['translation_text'] for result in hf_results]

    # 5. (분리 2) 그 외 텍스트: .apply()로 Google 번역 API 호출
    translated_other_list = []
    if not other_texts_series.empty:
        # googletrans의 .translate()는 객체를 반환하므로 .text로 접근
        translated_other_series = other_texts_series.apply(
            lambda text: gg_translator.translate(text, dest='en').text
        )
        translated_other_list = translated_other_series.tolist()

    # 6. 번역된 결과를 원래 인덱스를 가진 Series로 다시 만듭니다.
    translated_ko_series = pd.Series(translated_ko_list, index=ko_texts_series.index)
    translated_other_series = pd.Series(translated_other_list, index=other_texts_series.index)

    # 7. 두 결과를 합칩니다.
    all_translated_series = pd.concat([translated_ko_series, translated_other_series])

    # 8. unstack()으로 원래 DataFrame 형태로 복원합니다. (순서 보장!)
    translated_df_part = all_translated_series.unstack()

    # 9. 원본 DataFrame에 번역된 내용을 덮어씁니다.
    df_copy = df.copy() # 원본 수정을 피하기 위해 복사본 사용
    df_copy[cols_to_translate] = translated_df_part

    return df_copy

---

In [9]:
print("=== train_df 번역 시작 ===")
translated_train_df = batch_translate_dataframe(train_df, translator_hf, translator_gg)
display(translated_train_df)

=== train_df 번역 시작 ===
총 19435개 텍스트 중
  > 한국어 (HF 번역): 19177개
  > 그 외 (GG 번역): 258개


              id                  path  \
0     train_0001  train/train_0001.jpg   
1     train_0002  train/train_0002.jpg   
2     train_0003  train/train_0003.jpg   
3     train_0004  train/train_0004.jpg   
4     train_0005  train/train_0005.jpg   
...          ...                   ...   
3882  train_3883  train/train_3883.jpg   
3883  train_3884  train/train_3884.jpg   
3884  train_3885  train/train_3885.jpg   
3885  train_3886  train/train_3886.jpg   
3886  train_3887  train/train_3887.jpg   

                                               question  \
0     What's the location of the motor system in thi...   
1     What is the traditional Korean architecture in...   
2         What's the burn that you see in this picture?   
3        What is the bird that you see in this picture?   
4     What is it that people are looking at in this ...   
...                                                 ...   
3882        What kind of city is this photograph taken?   
3883     What's the col

In [11]:
print("\n=== test_df 번역 시작 ===")
translated_test_df = batch_translate_dataframe(test_df, translator_hf, translator_gg)
display(translated_test_df)


=== test_df 번역 시작 ===
총 19435개 텍스트 중
  > 한국어 (HF 번역): 19113개
  > 그 외 (GG 번역): 322개


             id                path  \
0     test_0001  test/test_0001.jpg   
1     test_0002  test/test_0002.jpg   
2     test_0003  test/test_0003.jpg   
3     test_0004  test/test_0004.jpg   
4     test_0005  test/test_0005.jpg   
...         ...                 ...   
3882  test_3883  test/test_3883.jpg   
3883  test_3884  test/test_3884.jpg   
3884  test_3885  test/test_3885.jpg   
3885  test_3886  test/test_3886.jpg   
3886  test_3887  test/test_3887.jpg   

                                               question  \
0                  What is the tower that you see here?   
1     What is the color of the leaves of the trees t...   
2               What is the building that you see here?   
3     What's the state of the traffic lights that yo...   
4               What's not on the chair in the picture?   
...                                                 ...   
3882  What are the features of the path that you see...   
3883               What is the tower that you see here?   


In [10]:
translated_train_df.head()

,id,path,question,a,b,c,d,answer
0,train_0001,train/train_0001.jpg,What's the location of the motor system in thi...,School yard.,Park,Inside the gym,Inside the mall.,b
1,train_0002,train/train_0002.jpg,What is the traditional Korean architecture in...,The palace.,Sex,The Aspects of the Secrets,Hanok,d
2,train_0003,train/train_0003.jpg,What's the burn that you see in this picture?,Three - wheeled motorcycles.,Motorcycle,Cars,Bicycle,a
3,train_0004,train/train_0004.jpg,What is the bird that you see in this picture?,Sparrows,Seagulls,Egret,duck,c
4,train_0005,train/train_0005.jpg,What is it that people are looking at in this ...,Sea Drilling,Sea Festival,Fireworks,Sunrise,d


In [12]:
translated_test_df.head()

,id,path,question,a,b,c,d
0,test_0001,test/test_0001.jpg,What is the tower that you see here?,63 Buildings,Namsan Tower,Lot de Tower,A River Bridge
1,test_0002,test/test_0002.jpg,What is the color of the leaves of the trees t...,Green,Yellow,Blue,Red
2,test_0003,test/test_0003.jpg,What is the building that you see here?,Apologize.,The palace photophony door.,The Dowsang Stone.,The end-of-the-end power.
3,test_0004,test/test_0004.jpg,What's the state of the traffic lights that yo...,Blue light,Blinking Fire,Red light.,Yellow light.
4,test_0005,test/test_0005.jpg,What's not on the chair in the picture?,A blanket.,Magazines,Laptop,Tigress.


In [13]:
translated_train_df.to_csv("train_en.csv")
translated_test_df.to_csv("test_en.csv")